In [1]:
# Import useful packages
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import pandas as pd
from astropy import units as u
from astropy.coordinates import Angle
from astropy.table import Table
from astropy.time import Time

In [2]:
# Read in the data

# Check which server we're on (in case the data is in different places on different servers)
import socket
hostname = socket.gethostname()

# Get paths to data
if hostname == "blpc1" or hostname == "blpc2":
    full_dataset_path = "/datax/scratch/nstieg/25GHz_higher.pkl"
    coherent_dataset_path = "/datax/scratch/nstieg/25GHz_higher_coherent.pkl"
    incoherent_dataset_path = "/datax/scratch/nstieg/25GHz_higher_incoherent.pkl"
elif hostname == "cosmic-gpu-1":
    full_dataset_path = "/mnt/cosmic-gpu-1/data0/nstiegle/25GHz_higher.pkl"
else:
    raise Exception("Data path not known")

# Read in data
# coherent = pd.read_pickle(coherent_dataset_path)
# incoherent = pd.read_pickle(incoherent_dataset_path)
df = pd.read_pickle(full_dataset_path)

# Save the human readable times
df["tstart_h"] = Time(df["tstart"], format="mjd").datetime

In [4]:
df.columns

Index(['id', 'beam_id', 'observation_id', 'tuning', 'subband_offset',
       'file_uri', 'file_local_enumeration', 'signal_frequency',
       'signal_index', 'signal_drift_steps', 'signal_drift_rate', 'signal_snr',
       'signal_coarse_channel', 'signal_beam', 'signal_num_timesteps',
       'signal_power', 'signal_incoherent_power', 'source_name', 'fch1_mhz',
       'foff_mhz', 'tstart', 'tsamp', 'ra_hours', 'dec_degrees',
       'telescope_id', 'num_timesteps', 'num_channels', 'coarse_channel',
       'start_channel', 'tstart_h'],
      dtype='object')

In [6]:
df['tsamp'][0]

0.524288

In [7]:
df['num_timesteps'][0]

16

In [9]:
# Figure out how long each 'pointing' or 'time segment' is 
df['pointing_length'] = df['tsamp'] * df['num_timesteps']

In [11]:
df['pointing_length'].unique()

array([8.388608, 4.194304, 2.097152])

In [12]:
df['num_timesteps'].unique()

array([16,  8,  4, 64])

In [13]:
df['tsamp'].unique()

array([0.524288, 0.131072])

In [20]:
.524288 * 4

2.097152

In [ ]:
# Look at how many incoherent, coherent, and phase_center sources we have
incoherent = df["source_name"] == "Incoherent"
tot_incoherent = incoherent.sum()
print("Incoherent:", tot_incoherent)
phase_center = df["source_name"] == "PHASE_CENTER"
tot_phase_center = phase_center.sum()
print("PHASE_CENTER:", tot_phase_center)
coherent = (df["source_name"] != "PHASE_CENTER") & (df["source_name"] != "Incoherent")
tot_coherent = coherent.sum()
print("all coherent:", tot_coherent)
print("Total:", tot_coherent + tot_phase_center + tot_incoherent)
# print("Matches:", num_rows)


Incoherent: 6503467
PHASE_CENTER: 21720703
all coherent: 2984740
Total: 31208910


NameError: name 'num_rows' is not defined

In [24]:
# Get the coherent points
df_coherent = df[coherent]

In [26]:
len(df_coherent)

2984740

In [27]:
df_coherent.columns

Index(['id', 'beam_id', 'observation_id', 'tuning', 'subband_offset',
       'file_uri', 'file_local_enumeration', 'signal_frequency',
       'signal_index', 'signal_drift_steps', 'signal_drift_rate', 'signal_snr',
       'signal_coarse_channel', 'signal_beam', 'signal_num_timesteps',
       'signal_power', 'signal_incoherent_power', 'source_name', 'fch1_mhz',
       'foff_mhz', 'tstart', 'tsamp', 'ra_hours', 'dec_degrees',
       'telescope_id', 'num_timesteps', 'num_channels', 'coarse_channel',
       'start_channel', 'tstart_h', 'pointing_length'],
      dtype='object')

In [30]:
df_by_sources = df_coherent.groupby('source_name')

In [33]:
sources = df_coherent['source_name'].unique
print(sources)

<bound method Series.unique of 0           3127348761205770496
1           3127348761205770496
2           3127348761205770496
3           3127348761205770496
4           3127348761205770496
                   ...         
31107165    3127348761205770496
31107166    3127348761205770496
31107167    3127348761205770496
31107168    3127348761205770496
31107169    3127348761205770496
Name: source_name, Length: 2984740, dtype: object>


In [55]:
for source, group in df_by_sources:
        total_observation_length = 0
        # Group into the start time
        pointings_dfs = group.groupby('tstart')
        for tstart, pointing_df in pointings_dfs:
            assert(len(pointing_df['pointing_length'].unique()) == 1)
            assert(len(pointing_df) > 0)
            total_observation_length += pointing_df['pointing_length'].iloc[0]
        print('source', source, "Total minutes analyzed by seticore:", total_observation_length / 60)

source 2535280716217508992 Total minutes analyzed by seticore: 3.774873600000001
source 2536546185381558272 Total minutes analyzed by seticore: 6.291455999999996
source 2542485953354555264 Total minutes analyzed by seticore: 10.450807466666653
source 2557518579407455104 Total minutes analyzed by seticore: 4.6137344
source 2646431411821304960 Total minutes analyzed by seticore: 16.8121685333333
source 3073619025268414208 Total minutes analyzed by seticore: 13.911108266666643
source 3074407546904630144 Total minutes analyzed by seticore: 15.658734933333303
source 3079227054261900160 Total minutes analyzed by seticore: 5.452595199999998
source 3113482167231459200 Total minutes analyzed by seticore: 1.1184810666666667
source 3113490855946843904 Total minutes analyzed by seticore: 1.1184810666666667
source 3113491272562268160 Total minutes analyzed by seticore: 1.1184810666666667
source 3113493127988068096 Total minutes analyzed by seticore: 1.1184810666666667
source 3127348761205770496 Tot

In [60]:
df_coherent.signal_drift_rate.max()

50.93170329928399